# Test the Retrained Face Recognition Model

In this notebook we test the ONNX model exported in notebook 03 on **still images** and **video**.

The retrained model has learned to distinguish *your* face from other faces:
- **Green** bounding boxes → your face (the class the model was trained on)
- **Red** bounding boxes → other / unknown faces

We'll run inference locally using the Ultralytics YOLO runtime with the ONNX backend, then move to server-side inference in notebook 05.

In [ ]:
# Uncomment if packages are not installed
# !pip install --no-cache-dir -r requirements.txt

from ultralytics import YOLO
import numpy as np
import cv2
from matplotlib import pyplot as plt
from pathlib import Path
from IPython.display import Video, display, Image as IPImage
import remote_infer

## Load the retrained ONNX model

In [ ]:
onnx_path = Path("runs/face-recognition/train/weights/best.onnx")
if not onnx_path.exists():
    print("Local trained ONNX not found. Using pre-trained model from HuggingFace...")
    from huggingface_hub import hf_hub_download
    onnx_path = hf_hub_download(
        repo_id="ariakang/YOLOv11n-face-detection",
        filename="model.onnx"
    )
    print("Note: This is a generic face detector, not your custom-trained model.")
    print("Run notebook 03 first to train a model that recognizes your face.")

model = YOLO(str(onnx_path), task="detect")
print(f"Model loaded from: {onnx_path}")

## Test on still images

In [ ]:
import glob

face_images = sorted(glob.glob("images/test_face_*.jpg"))
print(f"Testing on {len(face_images)} face images\n")

for image_path in face_images:
    results = model.predict(image_path, conf=0.5)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(8, 5))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

In [ ]:
group_images = sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing on {len(group_images)} group images\n")

for image_path in group_images:
    results = model.predict(image_path, conf=0.5)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(10, 6))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

## Video Inference — The Demo Highlight

Now for the fun part: we process a video where the model identifies **your face** versus other faces in real-time.

Each frame is run through the ONNX model locally. Your face should appear with a **green** bounding box,
while other faces get a **red** box. The annotated frames are reassembled into an output video.

In [ ]:
video_path = "videos/test_group_video.mov"

if Path(video_path).exists():
    print("Processing video — this may take a minute...")
    output_path = remote_infer.process_video_local(
        video_path, model, conf=0.5
    )
    print(f"\nAnnotated video saved to: {output_path}")
    display(Video(output_path, embed=True, width=640))
else:
    print(f"Video not found: {video_path}")
    print("Place a test video (10-30s, you + another person) in the videos/ folder.")

## Next Steps

The model recognizes faces correctly when running locally. Next, we'll **deploy it to RHOAI's model serving platform**
(KServe + OpenVINO Model Server) and query it via the **REST API** — the production inference pattern.

→ Open **notebook 05 — Query the Served Model**.